In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_INPUTS = True
REUSE_FANOUT_SEARCH = False
REUSE_VESSELNESS_EXTENSION = False
REUSE_FIGURES = False
REUSE_REPORT = False


# OpenPlaque — Secondary Endpoint Fan-Out

Starts at the accepted ~13.84 mm secondary-branch endpoint. Many 3-D directions are tested, but a direction must remain a compact coronary-like lumen from the first step through at least 2.4 mm before any longer vesselness extension is permitted. Research use only; no vessel identity is assigned automatically.


In [ ]:
!pip -q install scipy pandas matplotlib


In [ ]:
import os, shutil, sys
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
!git clone -q --depth 1 --branch secondary-endpoint-fanout-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
sys.path.insert(0, '/content/OpenPlaque/src')
!git -C /content/OpenPlaque rev-parse HEAD


In [ ]:
from IPython.display import display, Image
from openplaque.secondary_endpoint_fanout import synthetic_fanout_self_test
from openplaque.secondary_endpoint_fanout_v2 import SecondaryEndpointFanoutWorkflow

self_test = synthetic_fanout_self_test()
display(self_test)
assert self_test['passed'], self_test

wf = SecondaryEndpointFanoutWorkflow(
    root='/content/drive/MyDrive/OpenPlaque',
    reuse={
        'inputs': REUSE_INPUTS,
        'fanout_search': REUSE_FANOUT_SEARCH,
        'vesselness_extension': REUSE_VESSELNESS_EXTENSION,
        'figures': REUSE_FIGURES,
        'report': REUSE_REPORT,
    },
)
display(wf.cache_status())


In [ ]:
snapshot = wf.load_inputs()
display(snapshot)
display(wf.calibration)
print('Accepted endpoint z,y,x:', wf.anchor_point)
print('Fan-out directions:', 113)


In [ ]:
fan = wf.search_fanout(step_mm=0.30, strict_mm=2.4, max_mm=4.2)
display(fan.head(25))
print('Strict early survivors:', int(fan.strict_early_survivor.sum()))
print('Full 4.2-mm local survivors:', int(fan.full_local_survivor.sum()))
print('Maximum compact-lumen local length:', float(fan.local_length_mm.max()), 'mm')
print()
print('Terminal reasons:')
display(fan.terminal_reason.value_counts(dropna=False).rename_axis('reason').reset_index(name='directions'))


In [ ]:
summary = wf.run(top_local_survivors=3, max_cost=105.0)
display(summary)
print()
print('Local survivors:')
display(wf.local_survivors.head(20))
print()
print('Long-extension candidates:')
display(wf.extension_candidates if len(wf.extension_candidates) else wf.extension_candidates)


In [ ]:
names = wf.make_figures()
for name in names:
    path = str(wf.out / name)
    print(name)
    display(Image(filename=path))


In [ ]:
zip_path = wf.package()
print('Final ZIP:', zip_path)
print('Report:', wf.out / 'OPENPLAQUE_SECONDARY_ENDPOINT_FANOUT_REPORT.html')
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_SECONDARY_ENDPOINT_FANOUT_REPORT_BACK.zip')
